# WLS childhood-abuse application

This notebook is the single manuscript-facing analysis. It reads deterministic preprocessing outputs and validated cluster summaries; it does not run Gurobi or matching interactively. Historical values are shown only in a clearly labeled provenance section.

**Current status:** score preparation is corrected, but the sensitivity-value and runtime tables must be regenerated on the licensed cluster.

In [ ]:
from pathlib import Path
import ast
import json
import math
import numpy as np
import pandas as pd
try:
    from IPython.display import display, Markdown
except ModuleNotFoundError:
    display = print
    Markdown = str

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (cwd, *cwd.parents) if (p / 'application/data').exists())
APP = PROJECT_ROOT / 'application'
DERIVED = APP / 'data/derived'
LEGACY = APP / 'legacy_unvalidated'
OUTPUTS = APP / 'outputs'
pd.set_option('display.max_colwidth', 120)
print('Project root:', PROJECT_ROOT)

## 1. Deterministic preprocessing audit

In [ ]:
audit = json.loads((DERIVED / 'audit.json').read_text())
display(pd.DataFrame({
    'quantity': ['source rows', 'complete/matched rows', 'matched sets', 'outcomes'],
    'value': [audit['source_rows'], audit['matched_rows'], audit['number_matched_sets'], audit['number_outcomes']]
}))
display(pd.Series(audit['set_size_distribution'], name='number of sets').rename_axis('set size').to_frame())
print('Treatment counts:', audit['treatment_counts'])
print('Preliminary outcomes:', audit['preliminary_outcomes'])
print('Matching status:', audit['matching_status'])

In [ ]:
sentinel = pd.read_csv(DERIVED / 'covariate_sentinel_audit.csv')
display(sentinel[sentinel['negative_count'] > 0])
display(Markdown('The exact R script is still needed to verify how these WLS sentinel codes and the stated missingness indicators were handled during matching.'))

## 2. Corrected score construction and nominal p-values

In [ ]:
nominal = pd.read_csv(DERIVED / 'nominal_pvalues.csv')
nominal_display = nominal.copy()
nominal_display['p_value'] = nominal_display['p_value'].map(lambda value: f'{value:.4g}')
display(nominal_display)
assert nominal['selected_at_0.05'].sum() == 13
assert nominal.loc[nominal['selected_at_0.05'], 'outcome_index'].tolist() == [0, 1, 5, 6, 7, 9, 11, 14, 15, 16, 18, 19, 20]

In [ ]:
sample = pd.read_csv(DERIVED / 'analysis_sample.csv')
corrected_scores = pd.read_csv(DERIVED / 'application_scores.csv')
legacy_scores = pd.read_csv(LEGACY / 'scores/Whole_Qmat.csv')
correlations = pd.DataFrame({
    'quantity': ['raw alcohol/spouse', 'corrected score alcohol/spouse', 'legacy labeled alcohol/spouse'],
    'correlation': [
        sample[['alcohol', 'spouse']].corr().iloc[0, 1],
        corrected_scores[['alcohol', 'spouse']].corr().iloc[0, 1],
        legacy_scores[['alcohol', 'spouse']].corr().iloc[0, 1],
    ]
})
correlations_display = correlations.copy()
correlations_display['correlation'] = correlations_display['correlation'].map(lambda value: f'{value:.3f}')
display(correlations_display)
assert abs(correlations.loc[1, 'correlation']) < 0.1
assert correlations.loc[2, 'correlation'] > 0.95
display(Markdown('The legacy `spouse` score is a second alcohol transformation. Therefore all historical joint sensitivity values are quarantined.'))

## 3. Historical draft values — provenance only

In [ ]:
def parse_set(value):
    text = str(value).strip()
    if text.startswith('frozenset('):
        text = text[len('frozenset('):-1]
    return tuple(sorted(ast.literal_eval(text)))

legacy_full = pd.read_csv(LEGACY / 'results/gSval_Whole_size4.csv')
legacy_full['indices'] = legacy_full['chosen_cols_set'].map(parse_set)
expected = set(__import__('itertools').combinations(range(21), 4))
observed = set(legacy_full['indices'])
print(f'Legacy full-family coverage: {len(observed)} / {math.comb(21, 4)}')
print('Missing subsets:', sorted(expected - observed))

legacy_rows = legacy_full[legacy_full['indices'].isin([(0, 1, 5, 16), (1, 5, 14, 16)])].copy()
display(legacy_rows[['indices', 'gSval_half', 'gSval_naive_half', 'diff']])
print('These values reproduce the draft but are not corrected results.')

## 4. Corrected cluster results and manuscript summaries

In [ ]:
headline_path = OUTPUTS / 'headline_subsets.csv'
comparison_path = OUTPUTS / 'exact_vs_naive_summary.csv'
if headline_path.exists() and comparison_path.exists():
    headline = pd.read_csv(headline_path)
    comparison = pd.read_csv(comparison_path)
    display(headline.round(2))
    display(comparison)
else:
    display(Markdown('**Corrected Gurobi results are not present yet.** Run both cluster arrays, merge with full coverage checks, and execute `summarize_application.py`.'))

In [ ]:
runtime_path = OUTPUTS / 'runtime_table.csv'
if runtime_path.exists():
    runtime = pd.read_csv(runtime_path)
    assert runtime['decisions_equal'].all()
    display(runtime[['subset_label', 'gamma', 'miqcp_total_seconds', 'enumerative_total_seconds']])
else:
    display(Markdown('The corrected, equality-validated runtime table is not present yet.'))

## Final reproducibility checklist

- [x] Deterministic complete-case sample and corrected score matrix.
- [x] Pinned `matched_index.csv` validated structurally.
- [ ] Recover exact R matching, exposure-construction, weights, and balance-figure code.
- [ ] Merge all 715 corrected preliminary-family jobs.
- [ ] Merge all 5,985 corrected full-family jobs.
- [ ] Re-select R1 and R2 under the explicit rules.
- [ ] Validate enumerative/MIQCP equality and regenerate the runtime table.
- [ ] Replace every historical application number in the manuscript.